# Rainfall histograms: `_UH` stations vs `GML_SMO` stations

This notebook loads `raw_data/daily_wide_4302025.csv`, extracts daily rainfall values from the wide date columns, and compares distributions for:

- Stations whose `station_name` contains `_UH`
- Stations associated with `GML_SMO` (matched via `station_name`, `Organization`, or `source`)

The notebook also prints basic QC stats (missingness, zeros, negatives, min/max) so you can compare results with someone else.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path('..') / 'raw_data' / 'daily_wide_4302025.csv'
DATA_PATH

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head(3)

In [ ]:
# Identify which columns are dates (wide daily rainfall columns).
# In this file they look like MM/DD/YYYY.
date_cols = [c for c in df.columns if isinstance(c, str) and '/' in c and c.count('/') == 2]
meta_cols = [c for c in df.columns if c not in date_cols]
len(meta_cols), len(date_cols), meta_cols[:10], date_cols[:5]

In [ ]:
# Coerce station fields to string for consistent filtering
for col in ['station_name', 'Organization', 'source']:
    if col in df.columns:
        df[col] = df[col].astype('string')

uh_mask = df['station_name'].str.contains('_UH', na=False)
gml_mask = (
    df['station_name'].str.contains('GML_SMO', na=False)
    | df.get('Organization', pd.Series(False, index=df.index)).astype('string').eq('GML_SMO')
    | df.get('source', pd.Series(False, index=df.index)).astype('string').eq('GML_SMO')
)

print('Rows total:', len(df))
print('Rows _UH:', int(uh_mask.sum()))
print('Rows GML_SMO:', int(gml_mask.sum()))
print('Overlap rows:', int((uh_mask & gml_mask).sum()))
df.loc[uh_mask | gml_mask, ['station_name','Organization','source']].head(10)

In [ ]:
# Convert wide -> long for just the two groups (keeps this manageable).
use_mask = uh_mask | gml_mask
df_sub = df.loc[use_mask, meta_cols + date_cols].copy()

long = df_sub.melt(id_vars=meta_cols, value_vars=date_cols, var_name='date', value_name='rain')
long['date'] = pd.to_datetime(long['date'], format='%m/%d/%Y', errors='coerce')
long['rain'] = pd.to_numeric(long['rain'], errors='coerce')

long['group'] = np.where(long['station_name'].str.contains('_UH', na=False), '_UH', 'GML_SMO')
long[['station_name','date','rain','group']].head()

In [ ]:
def qc_summary(x: pd.Series) -> dict:
    x = pd.to_numeric(x, errors='coerce')
    n = len(x)
    nn = int(x.notna().sum())
    z = int((x == 0).sum())
    neg = int((x < 0).sum())
    return {
        'n_total': n,
        'n_nonnull': nn,
        'pct_nonnull': (nn / n) if n else np.nan,
        'pct_zero_among_nonnull': (z / nn) if nn else np.nan,
        'n_negative': neg,
        'min': float(np.nanmin(x.values)) if nn else np.nan,
        'p50': float(np.nanpercentile(x.values, 50)) if nn else np.nan,
        'p90': float(np.nanpercentile(x.values, 90)) if nn else np.nan,
        'p99': float(np.nanpercentile(x.values, 99)) if nn else np.nan,
        'max': float(np.nanmax(x.values)) if nn else np.nan,
    }

summary = (
    long.groupby('group')['rain']
    .apply(qc_summary)
    .apply(pd.Series)
)
summary

In [ ]:
# Histograms (all values including zeros)
bins = np.linspace(0, 200, 201)  # adjust if you want

fig, ax = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for i, g in enumerate(['_UH', 'GML_SMO']):
    vals = long.loc[long['group'] == g, 'rain'].dropna().values
    ax[i].hist(vals, bins=bins, alpha=0.8, color=('tab:blue' if g == '_UH' else 'tab:orange'))
    ax[i].set_title(f'{g}: histogram (0–200)')
    ax[i].set_xlabel('daily rainfall (raw units)')
    ax[i].set_ylabel('count')
plt.tight_layout()

In [ ]:
# Histograms for positive values only (often easier to compare shapes)
bins_pos = np.linspace(0, 100, 201)

fig, ax = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for i, g in enumerate(['_UH', 'GML_SMO']):
    vals = long.loc[(long['group'] == g) & (long['rain'] > 0), 'rain'].dropna().values
    ax[i].hist(vals, bins=bins_pos, alpha=0.8, color=('tab:blue' if g == '_UH' else 'tab:orange'))
    ax[i].set_title(f'{g}: positive-only histogram (0–100)')
    ax[i].set_xlabel('daily rainfall (raw units)')
    ax[i].set_ylabel('count')
plt.tight_layout()

In [ ]:
# Optional: log-scale view for positive rainfall
bins_log = np.logspace(-3, 2.5, 80)  # 0.001 to ~316

fig, ax = plt.subplots(figsize=(8, 5))
for g, color in [('_UH','tab:blue'), ('GML_SMO','tab:orange')]:
    vals = long.loc[(long['group'] == g) & (long['rain'] > 0), 'rain'].dropna().values
    ax.hist(vals, bins=bins_log, alpha=0.5, label=g, color=color)
ax.set_xscale('log')
ax.set_title('Positive rainfall (log-x)')
ax.set_xlabel('daily rainfall (raw units)')
ax.set_ylabel('count')
ax.legend()
plt.tight_layout()

In [ ]:
# If you suspect a units mismatch, compare distributions after a simple unit conversion.
# Example: inches -> mm (multiply by 25.4). Uncomment if needed.
# long['rain_mm_if_inches'] = long['rain'] * 25.4
# long.groupby('group')['rain_mm_if_inches'].apply(qc_summary).apply(pd.Series)